# 03 · Junção espacial foco × estação

Gera duas tabelas:

1. **`focos_com_clima`**: cada foco recebe os dados da estação INMET **mais próxima que tenha dado naquele dia**, dentro de 100 km (coluna `Dentro_raio_teste` marca os que também estão a até 50 km, para o teste de sensibilidade).
2. **`painel_estacao_dia`**: uma linha por estação × dia, com o número de focos em volta (inclusive **zeros**), para comparar dias com e sem fogo (subpergunta 4).

Observações: as variáveis de clima do INPE (`DiaSemChuva`, `Precipitacao`, `RiscoFogo`) continuam separadas das do INMET (prefixo `INMET_`). O bioma analisado é o do foco, não o da estação.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree
from google.colab import drive

drive.mount('/content/drive')

# ---- Parâmetros (decisões do README: raio principal 100 km, teste 50 km) ----
RAIO_KM = 100
RAIO_TESTE_KM = 50
R_TERRA_KM = 6371.0088

BASE = '/content/drive/MyDrive'
F_FOCOS = f'{BASE}/datasets_queimadas/focos_pantanal_cerrado_2020_2023.parquet'
F_INMET = f'{BASE}/datasets_inmet/processed/inmet_diario_pantanal_cerrado.parquet'
F_EST   = f'{BASE}/datasets_inmet/processed/estacoes_pantanal_cerrado.csv'
OUT_FOCOS  = f'{BASE}/datasets_queimadas/focos_com_clima.parquet'
OUT_PAINEL = f'{BASE}/datasets_queimadas/painel_estacao_dia.parquet'


## 1. Carga e conferência

In [ ]:
focos = pd.read_parquet(F_FOCOS)
inmet = pd.read_parquet(F_INMET)
est   = pd.read_csv(F_EST)

# Datas como dia (sem hora), mesmo padrão nas duas tabelas (UTC, sem fuso)
focos['Data'] = pd.to_datetime(focos['Data']).dt.normalize()
inmet['Data'] = pd.to_datetime(inmet['Data']).dt.normalize()

# Coordenadas são obrigatórias para a junção espacial
focos = focos.dropna(subset=['Latitude', 'Longitude']).reset_index(drop=True)
inmet = inmet.dropna(subset=['Latitude', 'Longitude']).reset_index(drop=True)

print('focos:', focos.shape, '| inmet:', inmet.shape, '| estações:', est.shape)
print('\nFocos por ano e bioma (conferir com o README: Cerrado 63.819 / 62.727 / 56.884 / 50.713; Pantanal 22.116 / 8.258 / 1.637 / 6.580):')
display(focos.groupby([focos['Data'].dt.year.rename('Ano'), 'Bioma']).size().unstack())

# Cada estação só pode ter uma linha por dia
assert not inmet.duplicated(['Codigo', 'Data']).any(), 'INMET com estação duplicada no mesmo dia'


## 2. Junção foco → estação mais próxima com dado no dia

In [ ]:
def vizinha_mais_proxima(focos, inmet):
    """Para cada foco, acha a estação MAIS PRÓXIMA que tem dado NAQUELE DIA.
    Devolve (posição da linha da estação em `inmet`, distância em km). -1 / NaN se não há estação com dado no dia.
    Pré-requisito: índices de focos e inmet são RangeIndex (reset_index)."""
    por_dia = {d: g for d, g in inmet.groupby('Data')}
    est_pos = np.full(len(focos), -1, dtype=np.int64)
    dist_km = np.full(len(focos), np.nan)
    coords_focos = np.radians(focos[['Latitude', 'Longitude']].to_numpy(dtype='float64'))

    for dia, pos in focos.groupby('Data').indices.items():
        g = por_dia.get(dia)
        if g is None or len(g) == 0:
            continue
        coords_est = np.radians(g[['Latitude', 'Longitude']].to_numpy(dtype='float64'))
        tree = BallTree(coords_est, metric='haversine')
        d, i = tree.query(coords_focos[pos], k=1)
        est_pos[pos] = g.index.to_numpy()[i[:, 0]]
        dist_km[pos] = d[:, 0] * R_TERRA_KM
    return est_pos, dist_km


COLS_CLIMA = ['Codigo', 'Estacao', 'UF', 'Precip_total', 'Temp_max', 'Temp_min', 'Temp_med',
              'Umid_min', 'Umid_med', 'Vento_max', 'Dias_sem_chuva']


def juntar_foco_estacao(focos, inmet, raio_km, raio_teste_km):
    est_pos, dist_km = vizinha_mais_proxima(focos, inmet)
    dentro = (est_pos >= 0) & (dist_km <= raio_km)          # NaN <= x é False

    clima = inmet.loc[est_pos[dentro], COLS_CLIMA].add_prefix('INMET_')
    clima.index = focos.index[dentro]

    out = focos.copy()
    out['Dist_estacao_km'] = dist_km
    out = out.join(clima)                                    # focos sem estação ficam com NaN
    out['Tem_clima'] = dentro
    out['Dentro_raio_teste'] = dentro & (dist_km <= raio_teste_km)
    return out


In [ ]:
focos_clima = juntar_foco_estacao(focos, inmet, RAIO_KM, RAIO_TESTE_KM)

# Nenhum foco pode ser perdido (as variáveis do INPE e do INMET ficam em colunas separadas)
assert len(focos_clima) == len(focos)
print('Focos com estação ≤', RAIO_KM, 'km no dia:', f"{focos_clima['Tem_clima'].mean():.1%}")
focos_clima.head()


## 3. Cobertura por bioma e sensibilidade (100 km vs 50 km)

In [ ]:
resumo = (focos_clima.groupby('Bioma')
          .agg(focos=('Bioma', 'size'),
               pct_com_clima_100km=('Tem_clima', 'mean'),
               pct_com_clima_50km=('Dentro_raio_teste', 'mean'),
               dist_mediana_km=('Dist_estacao_km', 'median'),
               dist_p90_km=('Dist_estacao_km', lambda s: s.quantile(0.9))))
resumo[['pct_com_clima_100km', 'pct_com_clima_50km']] = resumo[['pct_com_clima_100km', 'pct_com_clima_50km']].mul(100).round(1)
resumo['dist_mediana_km'] = resumo['dist_mediana_km'].round(1)
resumo['dist_p90_km'] = resumo['dist_p90_km'].round(1)
display(resumo)
# Esperado: o Pantanal tem cobertura pior (menos focos com estação perto) e distância mediana maior que o Cerrado.


## 4. Painel estação × dia (com dias de zero foco)

In [ ]:
# Bioma de cada ESTAÇÃO (só para montar o painel; a análise dos focos usa o Bioma do próprio foco)
dp, dc = est['dist_pantanal_km'], est['dist_cerrado_km']
est['Bioma_estacao'] = np.where(dp == 0, 'Pantanal', np.where(dc == 0, 'Cerrado', np.where(dp < dc, 'Pantanal', 'Cerrado')))


def contar(df, sufixo):
    c = (df.groupby(['INMET_Codigo', 'Data', 'Bioma']).size()
           .unstack('Bioma', fill_value=0)
           .reindex(columns=['Cerrado', 'Pantanal'], fill_value=0))
    c.columns = [f'n_focos_{sufixo}_{b}' for b in c.columns]
    c[f'n_focos_{sufixo}'] = c.sum(axis=1)
    return c.reset_index().rename(columns={'INMET_Codigo': 'Codigo'})


def montar_painel(inmet, est, focos_clima):
    base = (inmet[['Data', 'Codigo', 'Estacao', 'UF', 'Latitude', 'Longitude', 'Precip_total', 'Temp_max',
                   'Temp_med', 'Umid_min', 'Umid_med', 'Vento_max', 'Dias_sem_chuva']]
            .merge(est[['Codigo', 'Bioma_estacao']], on='Codigo', how='left'))
    c100 = contar(focos_clima[focos_clima['Tem_clima']], 'r100')
    c50  = contar(focos_clima[focos_clima['Dentro_raio_teste']], 'r50')
    painel = base.merge(c100, on=['Codigo', 'Data'], how='left').merge(c50, on=['Codigo', 'Data'], how='left')
    cols_n = [c for c in painel.columns if c.startswith('n_focos_')]
    painel[cols_n] = painel[cols_n].fillna(0).astype(int)      # dia sem foco = 0 (é isso que permite comparar dias com e sem fogo)
    painel['Teve_foco_r100'] = painel['n_focos_r100'] > 0
    painel['Teve_foco_r50']  = painel['n_focos_r50'] > 0
    return painel


painel = montar_painel(inmet, est, focos_clima)

# Conferências: todo foco com clima cai em exatamente uma linha estação × dia
assert painel['n_focos_r100'].sum() == focos_clima['Tem_clima'].sum()
assert painel['n_focos_r50'].sum() == focos_clima['Dentro_raio_teste'].sum()
print(painel.shape, '| % de estação-dia com algum foco (100 km):', f"{painel['Teve_foco_r100'].mean():.1%}")
painel.head()


## 5. Salvar

In [ ]:
focos_clima.to_parquet(OUT_FOCOS, index=False)
painel.to_parquet(OUT_PAINEL, index=False)
print('Salvos em:', OUT_FOCOS, 'e', OUT_PAINEL)
